<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">模块 14 · Lab 14</div><div style="color:#17212b;font-size:30px;font-weight:750">资源隔离与查询保护：让看板不再被临时分析拖慢</div><p style="color:#475569;line-height:1.7">本实验把看板和临时分析拆进两个 Workload Group，用多个会话同时提交慢查询，观察执行、排队和拒绝；再用执行时限和 SQL Block Rule 挡住失控查询，最后回收授权并清理。</p></div>

## 你将完成什么

- 确认没有配置时查询落在 normal 组；
- 创建看板组和临时分析组，只把看板组的 USAGE_PRIV 授予课程角色；
- 同时提交多条慢查询，观察执行、排队、拒绝和绕过排队；
- 区分 queue_timeout 和 query_timeout 的报错；
- 用 SQL Block Rule 在执行前拦截不带日期条件的全表扫描；
- 按顺序回收授权、删除角色和组，确认环境已清理。

## 前置条件和安全边界

**前置条件：** 本实验不依赖 Level 1 的业务表，只需要课程沙箱连接。

**安全边界：** 只操作 orders_monthly_l3、course_dashboard_l3、course_adhoc_l3、course_dashboard_reader_l3 和 course_scan_guard_l3；不修改 normal 组和任何账号属性，不创建真实用户，不打印密码。全局 SQL Block Rule 只存在于一个单元格内，无论成功与否都会在这个单元格结束前删除；如果这个单元格中途被中断或内核被重启，重新运行第 1 步即可清理。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.workload import BackgroundQuery, group_load, run_query, show_outcomes, wait_for_group

lab = connect_sandbox()

DASHBOARD = "course_dashboard_l3"
ADHOC = "course_adhoc_l3"

## 0. 当前查询落在哪个组？

选组顺序是：会话变量 workload_group、账号属性 default_workload_group、内置的 normal 组。先确认课程连接的会话变量为空、账号默认组是 normal，再记下 normal 组当前的配置，最后用它确认本实验没有修改 normal。

In [ ]:
lab.sql("SELECT @@workload_group AS session_workload_group", title="会话变量 workload_group")
lab.sql("SHOW PROPERTY LIKE 'default_workload_group'", title="账号默认组")
lab.sql("SHOW WORKLOAD GROUPS", title="现有的 Workload Group")

NORMAL_BEFORE = lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'")

**路由验收：** 会话变量为空，账号默认组为 normal，所以这个连接上的查询都落在 normal。如果你以前把 root 的 default_workload_group 改成了别的组，这里会失败；本实验不替你修改账号属性，请先自行改回 normal。

In [ ]:
expect(lab.query("SELECT @@workload_group"), [("",)], title="会话变量为空")
expect(lab.query("SHOW PROPERTY LIKE 'default_workload_group'"), [("default_workload_group", "normal")], title="账号默认组为 normal")

## 1. 准备实验表并清理遗留对象

Workload Group、角色和 SQL Block Rule 都是集群级对象，删除课程库时不会随之删除。初始化先按“规则、角色、组”的顺序清理上次遗留的同名对象，再重建按月分区的 orders_monthly_l3，保证整本 Notebook 可以重复执行。三个分区沿用 Module 13 的边界，每个月一笔订单。

In [ ]:
lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")

lab.execute("DROP TABLE IF EXISTS orders_monthly_l3")
lab.execute("""
CREATE TABLE orders_monthly_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01"),
    PARTITION p202503 VALUES LESS THAN ("2025-04-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("orders_monthly_l3", ["order_date", "order_id", "amount"], [("2025-01-15", 140001, 100), ("2025-02-15", 140002, 200), ("2025-03-15", 140003, 300)])
lab.sql("SELECT * FROM orders_monthly_l3 ORDER BY order_date, order_id", title="按月分区的实验表");

**初始结果验收：** 应有 3 行，分别落在 p202501、p202502、p202503。第 3 步的慢查询只读取 3 月的一行，第 5 步的扫描规则用到全部三个分区。

In [ ]:
expect(
    lab.query("""
SELECT CAST(order_date AS STRING), order_id, amount
FROM orders_monthly_l3
ORDER BY order_date, order_id
"""),
    [
        ("2025-01-15", 140001, "100.00"),
        ("2025-02-15", 140002, "200.00"),
        ("2025-03-15", 140003, "300.00"),
    ],
    title="三个月份各一笔订单",
)

## 2. 创建看板组和临时分析组

两个组的参数与正文 14.5 的资源 Runbook 一致。临时分析组的 max_concurrency 和 max_queue_size 都设为 1，这样几条查询就能看到排队和拒绝；它的 max_cpu_percent 在沙箱中不生效，因为 BE 没有配置 CPU cgroup。授权只给课程角色，而且只授予看板组的 USAGE_PRIV。

In [ ]:
lab.execute("""
CREATE WORKLOAD GROUP course_dashboard_l3
PROPERTIES (
    "max_concurrency" = "8",
    "max_queue_size" = "20",
    "queue_timeout" = "10000",
    "max_memory_percent" = "50"
)
""")
lab.execute("""
CREATE WORKLOAD GROUP course_adhoc_l3
PROPERTIES (
    "max_concurrency" = "1",
    "max_queue_size" = "1",
    "queue_timeout" = "30000",
    "max_memory_percent" = "30",
    "max_cpu_percent" = "30"
)
""")
lab.execute("CREATE ROLE course_dashboard_reader_l3")
lab.execute("GRANT USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' TO ROLE 'course_dashboard_reader_l3'")
lab.sql("SHOW WORKLOAD GROUPS LIKE 'course%'", title="课程的两个 Workload Group");

**资源组验收：** 两个组的并发、队列、等待时间和内存上限与设计一致；课程角色只有看板组的 Usage_priv，没有临时分析组的权限。

In [ ]:
expect(
    lab.query("""
SELECT NAME, MAX_CONCURRENCY, MAX_QUEUE_SIZE, QUEUE_TIMEOUT, MAX_MEMORY_PERCENT
FROM information_schema.workload_groups
WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY NAME
"""),
    [
        ("course_adhoc_l3", 1, 1, 30000, "30%"),
        ("course_dashboard_l3", 8, 20, 10000, "50%"),
    ],
    title="两个组的上限",
)
expect(
    lab.query("""
SELECT GRANTEE, WORKLOAD_GROUP_NAME, PRIVILEGE_TYPE
FROM information_schema.workload_group_privileges
WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY GRANTEE, WORKLOAD_GROUP_NAME
"""),
    [("course_dashboard_reader_l3", "course_dashboard_l3", "Usage_priv")],
    title="课程角色只能使用看板组",
)

**阅读授权结果：** USAGE_PRIV 只表示“可以使用这个组”，不会让查询自动落进去。真实的 BI 账号还要由管理员设置 default_workload_group，查询才会默认进入看板组；本实验不修改账号属性，后面的会话用会话变量 workload_group 指定组。

## 3. 同时提交五条查询：执行、排队还是拒绝？

每条查询用 sleep() 模拟一次慢扫描，只读取 3 月的一行，所以 sleep(6) 大约执行 6 秒。每条查询各自打开一个会话，先关闭 SQL Cache，再执行 SET workload_group 等会话变量，然后提交查询。关闭 SQL Cache 是因为命中缓存的查询会直接返回上次的结果，不进入队列，实验就看不到排队了。五条查询依次是：

- A 进入临时分析组，占住唯一的执行名额；
- B 进入临时分析组，排在 A 后面；
- C 进入临时分析组，队列已满，应当立即被拒绝；
- D 进入看板组，使用看板组自己的名额；
- E 进入临时分析组，但会话设置了 bypass_workload_group。

为了避免时序抖动，每一步都先等组内的执行和排队计数达到预期，再提交下一条。

In [ ]:
SLOW = "SELECT order_id, sleep({seconds}) AS slow_scan FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'"

a = BackgroundQuery(lab, "A 临时分析", SLOW.format(seconds=6), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = BackgroundQuery(lab, "B 临时分析", SLOW.format(seconds=1), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=1)

lab.sql("""
SELECT q.QUERY_STATUS, g.NAME AS workload_group, q.QUEUE_START_TIME, q.QUEUE_END_TIME
FROM information_schema.active_queries q
JOIN information_schema.workload_groups g ON q.WORKLOAD_GROUP_ID = g.ID
WHERE g.NAME = 'course_adhoc_l3'
ORDER BY q.QUERY_STATUS
""", title="A 在执行，B 在排队");

In [ ]:
c = run_query(lab, "C 临时分析", SLOW.format(seconds=1), workload_group=ADHOC)
d = run_query(lab, "D 看板", SLOW.format(seconds=1), workload_group=DASHBOARD)
e = run_query(lab, "E 绕过排队", SLOW.format(seconds=1), workload_group=ADHOC, bypass_workload_group=True)
load_while_a_runs = group_load(lab, ADHOC)

a_outcome, b_outcome = a.result(), b.result()
show_outcomes("五条查询的结果", [a_outcome, b_outcome, c, d, e])
wait_for_group(lab, ADHOC, running=0, waiting=0);

**排队验收：** A、B、D、E 成功，C 被拒绝；C、D、E 结束时临时分析组仍是 1 条执行、1 条排队；B 的耗时明显长于 D，因为它先排队等 A 结束。

In [ ]:
expect(
    [(o.label, o.result) for o in (a_outcome, b_outcome, c, d, e)],
    [
        ("A 临时分析", "OK"),
        ("B 临时分析", "OK"),
        ("C 临时分析", "query waiting queue is full, queue capacity=1, waiting num=1"),
        ("D 看板", "OK"),
        ("E 绕过排队", "OK"),
    ],
    title="执行、排队、拒绝和绕过排队",
)
expect(load_while_a_runs, {"running": 1, "waiting": 1}, title="C、D、E 结束时临时分析组的计数")
expect(b_outcome.seconds > d.seconds + 3, True, title="B 的耗时包含排队时间")

**阅读结果：** active_queries 中 A 的 QUERY_STATUS 是 RUNNING；B 是 WAIT_IN_QUEUE，还没有 QUEUE_END_TIME，因为它还没拿到名额。C 被拒绝是因为临时分析组的 max_queue_size 为 1，B 已经占了唯一的排队位置，报错中的 queue capacity 和 waiting num 就是这两个数。D 属于看板组，有自己的名额，不受临时分析组影响。E 和 C 在同一个组，却因为会话设置了 bypass_workload_group 而跳过了排队；这个设置不需要任何权限，所以排队只是约定，不是隔离边界。

## 4. 分别限制等待时间和执行时间

queue_timeout 限制在队列里等多久，query_timeout 限制执行多久。先把临时分析组的 queue_timeout 缩短为 1 秒，重复“A 占住名额、B 排队”；再在另一个会话里把 query_timeout 设为 2 秒，执行一条需要 4 秒的查询 F。F 没有指定组，落在 normal。这一步会改动临时分析组的 queue_timeout，之后想重新观察第 3 步，请从第 1 步开始重新运行。

In [ ]:
lab.execute('ALTER WORKLOAD GROUP course_adhoc_l3 PROPERTIES ("queue_timeout" = "1000")')

a = BackgroundQuery(lab, "A 临时分析", SLOW.format(seconds=3), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = run_query(lab, "B 排队超时", SLOW.format(seconds=1), workload_group=ADHOC)
f = run_query(lab, "F 执行超时", SLOW.format(seconds=4), query_timeout=2)

a_outcome = a.result()
show_outcomes("两种时限", [a_outcome, b, f])
wait_for_group(lab, ADHOC, running=0, waiting=0);

**时限验收：** A 成功；B 排队约 1 秒后报 query queue timeout；F 执行约 2 秒后报 query timeout。

In [ ]:
expect(
    [(o.label, o.result) for o in (a_outcome, b, f)],
    [
        ("A 临时分析", "OK"),
        ("B 排队超时", "query queue timeout, timeout: 1000 ms"),
        ("F 执行超时", "query timeout"),
    ],
    title="两种超时报错",
)
expect(b.seconds < 2, True, title="B 排队约 1 秒后放弃")
expect(1.5 <= f.seconds < 4, True, title="F 执行约 2 秒后被取消")

**阅读结果：** 两条报错对应不同的阶段。query queue timeout 表示查询一直没有拿到名额，没有在 BE 上执行；query timeout 表示查询已经执行了 2 秒才被取消，这段时间的资源已经用掉了。query_timeout 是会话变量，账号可以自己调大；生产环境要由管理员用 SET PROPERTY FOR 为账号设置，账号属性优先于会话变量。

## 5. 用 SQL Block Rule 挡住全表扫描

下面的全局规则限制单次查询最多扫描 2 个分区。不带日期条件的查询要扫描 3 个分区，应当被拒绝；只查 3 月的看板查询只扫描 1 个分区，照常执行。全局规则对沙箱里的所有会话生效，包括当前的 root 连接，所以创建、验证和删除都放在同一个单元格里，用 try/finally 保证规则一定被删除。

演示用 SUM(amount)，不用 COUNT(*)：正文 14.4 说明了不带条件的 COUNT(*) 可能不扫描任何分区。

In [ ]:
lab.execute("""
CREATE SQL_BLOCK_RULE course_scan_guard_l3
PROPERTIES ("partition_num" = "2", "global" = "true", "enable" = "true")
""")
try:
    lab.sql("SHOW SQL_BLOCK_RULE FOR course_scan_guard_l3", title="扫描规则")
    full_scan = run_query(lab, "全表 SUM", "SELECT SUM(amount) FROM orders_monthly_l3")
    one_month = run_query(lab, "只查 3 月", "SELECT SUM(amount) FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'")
finally:
    lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")

show_outcomes("扫描规则的拦截结果", [full_scan, one_month])

**拦截验收：** 全表查询命中 course_scan_guard_l3，只查一个月的查询照常执行；单元格结束后规则已经不存在。

In [ ]:
expect(
    [(o.label, o.result) for o in (full_scan, one_month)],
    [
        ("全表 SUM", "sql hits sql block rule: course_scan_guard_l3, reach partition_num : 2"),
        ("只查 3 月", "OK"),
    ],
    title="扫描规则只拦截全表扫描",
)
expect(
    [row for row in lab.query("SHOW SQL_BLOCK_RULE") if row[0] == "course_scan_guard_l3"],
    [],
    title="全局规则已删除",
)

**阅读结果：** 被拒绝的查询没有执行，报错中给出规则名和上限，Runbook 要记下规则名，方便被拦截的人找到原因。partition_num : 2 是规则的上限，不是实际扫描的分区数。只查 3 月的查询经过分区裁剪只扫描 1 个分区，不受影响。

## 6. 回收授权，删除角色和组

撤销顺序与正文 14.5 一致。先确认影响范围：没有 Workload Policy 引用这两个组；本实验没有修改账号属性，所以也没有账号把它们设为 default_workload_group。然后回收 USAGE_PRIV，删除角色，最后删除两个组。orders_monthly_l3 留给你继续查询，重新运行第 1 步时会重建。

In [ ]:
expect(
    lab.query("""
SELECT COUNT(*) FROM information_schema.workload_policy
WHERE WORKLOAD_GROUP IN ('course_dashboard_l3', 'course_adhoc_l3')
"""),
    [(0,)],
    title="没有 Workload Policy 引用课程的组",
)
lab.execute("REVOKE USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' FROM ROLE 'course_dashboard_reader_l3'")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")
lab.sql("SHOW WORKLOAD GROUPS", title="清理后的 Workload Group");

**清理验收：** 课程的两个组、角色和组权限都已删除，扫描规则已在第 5 步删除；normal 组的配置与第 0 步记录的一致。

In [ ]:
expect(
    lab.query("""
SELECT
    (SELECT COUNT(*) FROM information_schema.workload_groups
     WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')),
    (SELECT COUNT(*) FROM information_schema.workload_group_privileges
     WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3'))
"""),
    [(0, 0)],
    title="课程的组和组权限已删除",
)
expect([row[0] for row in lab.query("SHOW ROLES") if row[0] == "course_dashboard_reader_l3"], [], title="课程角色已删除")
expect(lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'"), NORMAL_BEFORE, title="normal 组未被修改")

## 独立练习：为看板上线写一份资源配置清单

BI 团队有 5 个账号，高峰期大约 20 个看板查询同时刷新；分析团队有 3 个账号，查询不定时、范围不确定；集群有 3 个 FE。请写出：

1. 看板组和临时分析组的并发、队列和等待时间；
2. 两类账号怎样落到各自的组；
3. 分析师在队列满、排队超时和执行超时时看到的报错；
4. 怎样拦住不带日期条件的全表扫描，它对谁生效；
5. 撤销整套配置的顺序。

先在下面写出你的判断，再展开参考解释。

In [ ]:
# TODO: 写下两个组的参数、账号路由、保护规则和撤销顺序。

<details>
<summary>参考解释</summary>

队列按 FE 分别计算，看板组的并发要按 3 个 FE 折算，并确认连接能在 FE 之间均衡；队列留出刷新突发的余量，等待时间设为几秒。临时分析组的并发和队列都有意设得很小，宁可让大查询看到“队列已满”，也不让它们在 BE 上堆积。BI 账号通过角色获得看板组的 USAGE_PRIV，由管理员设置 default_workload_group；分析师只获得临时分析组的 USAGE_PRIV。三种报错依次是 query waiting queue is full、query queue timeout 和 query timeout，执行时限要由管理员用账号属性设定。全表扫描用限制 partition_num 的 SQL Block Rule 拦截，只绑定到分析师账号；设为 global 会同时影响看板和管理员。撤销时先删除引用这些组的 Workload Policy，把账号的默认组改回 normal，再回收 USAGE_PRIV，删除角色、组和规则，最后用 SHOW 语句核实。

</details>

## 实验回顾

- 查询按会话变量、账号属性、normal 的顺序选组；
- max_concurrency、max_queue_size 和 queue_timeout 决定执行、排队还是拒绝，看板组不受临时分析组排满的影响；
- bypass_workload_group 人人可设，只跳过排队，所以排队不是隔离边界；
- queue_timeout 在执行前放弃，query_timeout 在执行中取消，SQL Block Rule 在执行前拒绝；
- 删除组之前先回收授权，避免同名组重建后旧授权重新生效。